# SRD Honest Benchmark — Colab Runner

Runs the SRD perplexity sweep on TinyLlama-1.1B and produces the data behind `docs/SRD_RESULTS.md`.

**Pipeline:** install → unit tests → coherence smoke test → SRD perplexity sweep → K-quant baseline → plot → download.

**Hardware:** Colab T4 (free tier). Total wall-clock ~30-45 min including model download. Most cells are fast; the SRD sweep (Cell 5) is the long pole at ~10-15 min.

**Deliverables this notebook produces (downloaded in Cell 8):**
- `srd_sweep.json` — rows 1-5 of the results table (FP16 + 4 SRD configs)
- `kquant_sweep.json` — rows 6-9 (K-quant baselines, cited)
- `srd_perplexity_vs_bpw.png` — the headline plot

Paste the two JSON files back into the chat and the write-up at `docs/SRD_RESULTS.md` gets filled in.

**Source:** github.com/Orivael-Dev/axiom · plan at `/root/.claude/plans/abundant-wandering-salamander.md`

In [ ]:
# Cell 1: Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Sweep will run on CPU (~3 hours instead of ~15 min).")
    print("Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Clone the AXIOM repo + install research/quant requirements
#
# GIT_REF points at the SRD prototype branch. Change to 'main' once
# the PR is merged.
import os
GIT_REF = "claude/srd-prototype-benchmark-JRtv1"

!rm -rf axiom
!git clone --depth 1 --branch {GIT_REF} https://github.com/Orivael-Dev/axiom.git
os.chdir("axiom")
!pip install -q -r research/quant/requirements.txt
print("\n[setup] ready. branch:", GIT_REF)


In [ ]:
# Cell 3: Phase A — unit tests (no model download, ~2 sec)
#
# Confirms the kernel reproduces the same numbers the unit tests pin:
#   - bpw matches hand calc (4 + 8 + 32/G + 32/G)
#   - residue strictly improves MSE
#   - shape / dtype / value-range invariants hold
# If any of these fail, halt — the sweep below is meaningless until
# the kernel is right.
import os
os.environ.setdefault("AXIOM_MASTER_KEY", "colab_smoke_key_only")
!python -m pytest tests/test_axiom_quant.py -q

In [ ]:
# Cell 4: Phase B — coherence smoke test
#
# Downloads TinyLlama-1.1B (~2 GB cached after first run), applies
# SRD at alpha=1, generates 80 tokens. The output should be coherent
# English. If the model spits gibberish at alpha=1, the kernel is
# bug-bait — STOP HERE and re-check axiom_quant.py.
#
# At alpha=0 the output should be noticeably degraded but still
# English-shaped (4-bit only is lossier than the published
# Q4_K_M numbers because there's no per-block scale tuning).
!python research/quant/quantize_model.py \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --alpha 1.0 \
    --prompt "Once upon a time, in a small village, " \
    --tokens 60

In [ ]:
# Cell 5: Phase C1 — SRD perplexity sweep (rows 1-5)
#
# Configs run: FP16 baseline, SRD alpha={0, 0.5, 1.0} per-block g=64,
# SRD alpha=1.0 per-tensor. Each config gets fresh weights so the
# previous run can't contaminate the next. Sliding-window PPL with
# stride=512, context=2048 (standard llama.cpp conventions).
#
# Sanity warning fires if the FP16 baseline drifts >0.5 from the
# published TinyLlama PPL (~7.7) — that means the eval harness is
# broken, not the kernel.
#
# Wall-clock: ~10-15 min on T4. The dataset fingerprint guard
# writes results/wikitext_fingerprint.txt on first run.
!python -m research.quant.bench_perplexity \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --group-size 64 \
    --output research/quant/results/srd_sweep.json

In [ ]:
# Cell 6: Phase C2 — K-quant baseline (rows 6-9)
#
# Cites the published llama.cpp PPLs for TinyLlama-1.1B at
# Q4_K_M / Q5_K_M / Q6_K / Q8_0. Fast (~1 sec, no download).
# Stride convention may differ from our SRD eval — see write-up.
#
# For stride-matched apples-to-apples, run Cell 6A then Cell 6B
# instead of this cell. They build llama.cpp from source (cmake)
# and run llama-perplexity --ppl-stride 512. ~30 min on H100.

!python -m research.quant.bench_llamacpp \
    --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --output research/quant/results/kquant_sweep.json
print("Done — source=published_cite (stride caveat applies)")


In [ ]:
# Cell 6A: Build llama.cpp from source (~5 min on H100)
# Run once per Colab session, then run Cell 6B.
# Skip this cell if you only want the cited numbers (run Cell 6 instead).
import subprocess, os

BUILD_ROOT = "/opt/llama.cpp"
BIN_PATH   = f"{BUILD_ROOT}/build/bin"

if os.path.exists(f"{BIN_PATH}/llama-perplexity"):
    print(f"Already built: {BIN_PATH}/llama-perplexity")
else:
    print("Cloning llama.cpp (depth 1)...")
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/ggml-org/llama.cpp", BUILD_ROOT], check=True)
    print("Configuring cmake (CUDA on)...")
    subprocess.run(["cmake", BUILD_ROOT, f"-B{BUILD_ROOT}/build",
                     "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release",
                     "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF",
                     "-DLLAMA_BUILD_EXAMPLES=OFF", "-DBUILD_SHARED_LIBS=OFF"], check=True)
    ncpu = str(os.cpu_count() or 4)
    print(f"Building with {ncpu} cores (takes ~5 min)...")
    # Only need llama-perplexity — GGUFs are downloaded from HF Hub,
    # so llama-quantize is not required.
    subprocess.run(["cmake", "--build", f"{BUILD_ROOT}/build",
                     "--config", "Release", "-j", ncpu,
                     "--target", "llama-perplexity"], check=True)
    print("Build done.")

LLAMA_BIN = BIN_PATH
assert os.path.exists(f"{LLAMA_BIN}/llama-perplexity"), f"binary not found at {LLAMA_BIN}"
print(f"OK: {LLAMA_BIN}/llama-perplexity")


In [ ]:
# Cell 6B: K-quant benchmark with stride-matched eval (~25 min on H100)
# Requires Cell 6A (sets LLAMA_BIN).
#
# Downloads pre-quantized GGUFs from TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF
# (no HF→GGUF conversion needed — avoids convert script version issues).
# The output JSON will have source=rerun_local and stride=512.
import subprocess, sys, os
from datasets import load_dataset

print("Writing WikiText-2 test split to /tmp/wt2.txt...")
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wt2_path = "/tmp/wt2.txt"
open(wt2_path, "w").write("\n\n".join(ds["text"]))
print(f"Written {os.path.getsize(wt2_path) // 1024} KB")

print("Running K-quant benchmark (downloads GGUFs, --ppl-stride 512, ~25 min on H100)...")
subprocess.run([
    sys.executable, "-m", "research.quant.bench_llamacpp",
    "--rerun-locally",
    "--llama-bin",     LLAMA_BIN,
    "--gguf-dir",      "/tmp/srd_gguf",
    "--wikitext-file", wt2_path,
    "--ppl-stride",    "512",
    "--output",        "research/quant/results/kquant_sweep.json",
], check=True)
print("Done — check kquant_sweep.json: source=rerun_local, stride=512")


In [ ]:
# Cell 7: Phase E — generate the perplexity-vs-bpw plot
import os
os.makedirs("docs", exist_ok=True)
!python -m research.quant.plot_results \
    --inputs research/quant/results/srd_sweep.json,research/quant/results/kquant_sweep.json \
    --output docs/srd_perplexity_vs_bpw.png
from IPython.display import Image
Image("docs/srd_perplexity_vs_bpw.png")

In [ ]:
# Cell 8: Show the verdict + download artifacts
import json
from pathlib import Path

srd = json.load(open("research/quant/results/srd_sweep.json"))
kq  = json.load(open("research/quant/results/kquant_sweep.json"))

print("─── SRD sweep ──────────────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}")
for r in srd:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}")

print("\n─── K-quant baseline ───────────────────────────────")
print(f"{'name':<32} {'bpw':>6} {'PPL':>8}  source")
for r in kq:
    print(f"{r['name']:<32} {r['bpw_reported']:>6.2f} {r['perplexity']:>8.3f}  {r.get('source', '?')}")

print("\n─── Verdict (pre-committed decision rule) ──────────")
srd_a1 = next((r for r in srd if r['name'].startswith('srd_alpha_1.0_g')), None)
q6 = next((r for r in kq if 'Q6_K' in r['name']), None)
if srd_a1 and q6:
    margin = q6['perplexity'] - srd_a1['perplexity']
    print(f"SRD α=1.0 @ {srd_a1['bpw_reported']:.2f} bpw: PPL = {srd_a1['perplexity']:.3f}")
    print(f"Q6_K     @ {q6['bpw_reported']:.2f} bpw: PPL = {q6['perplexity']:.3f}")
    print(f"Margin (Q6_K - SRD): {margin:+.3f}")
    if margin >= 0.05:
        print("→ VERDICT: SRD beats Q6_K by ≥0.05 PPL. Worth pursuing per the plan.")
    elif margin > 0:
        print("→ VERDICT: SRD edges Q6_K but inside the 0.05 noise margin. Not worth the ~2× memory.")
    else:
        print("→ VERDICT: SRD loses to Q6_K at lower bpw. Shelve.")

# Trigger Colab download — works in Colab only.
try:
    from google.colab import files
    for path in ["research/quant/results/srd_sweep.json",
                 "research/quant/results/kquant_sweep.json",
                 "docs/srd_perplexity_vs_bpw.png"]:
        if Path(path).exists():
            files.download(path)
except ImportError:
    print("\n(google.colab not available — artifacts left at the paths above)")